# 11 · A model that decides

**Course:** Building LLM Agents, extra notebook (after 10)
**Where we are:** you've built routers, critics and guardrails, and every one of them was a chat model asked to *write* a verdict that your code then parsed. Your course key now reaches a second, very different model that makes those decisions directly. This notebook is about how it works, and when to reach for it.

### What you'll learn
- What a *decision model* is, and how it differs from the chat model you've used so far
- How to ask Clef questions: a `state`, plus `noul`, `choice` and `score` questions
- How to read the answers: a probability for every option, the pick, and a confidence
- Three jobs it does well inside an agent: router (notebook 08), critic gate (09) and guardrail (06)
- When not to trust it, and how to fall back to the chat model

### Prerequisites
- Notebooks 01 to 04, plus the router in 08, the reflection loop in 09 and the sandbox in 06
- Your `.env` is set up and `from helpers import ...` works

> Mental model: a chat model *writes*; a decision model *chooses*. You describe a situation and ask questions with fixed answers. Clef reads everything once and returns a probability for every answer you allowed. It cannot say anything you didn't ask, which is exactly what you want from a component whose output feeds an `if`.

> Recap of 08 and 09. `classify()` asked Qwen for `{"category": ...}` in JSON mode, checked the label against the list, and fell back to `general` when it wasn't one of ours. `critique()` asked Qwen for `{"good": ..., "feedback": ...}`. Both are decisions dressed up as writing. Keep them in mind: we'll rebuild each one with Clef and compare.

## 0. Setup

In [ ]:
import json
import time

from openai import APIError, BadRequestError
from helpers import client, MODEL, CLEF_MODEL, available_models, ask, jprint

print("Chat model    :", MODEL)
print("Decision model:", CLEF_MODEL)

## 1. Two models on one key

Ask the endpoint what your key can reach, as in notebook 01:

In [ ]:
available_models()

Two names. `Qwen3.8-27B` is the chat model behind every notebook so far. `clef-flash-not-for-production` is Clef: Cloudflare's Clef-Flash, a 9.4-billion-parameter model built on Qwen3.5-9B, with a head that scores answers in place of one that generates text. `helpers.py` never picks it as `MODEL`, because it can't chat.

The name is honest, too. It's a trial service on one shared V100 GPU, not something to build production on. Section 9 comes back to that.

## 2. It doesn't chat

Try talking to it the way you'd talk to Qwen:

In [ ]:
try:
    client.chat.completions.create(
        model=CLEF_MODEL,
        messages=[{"role": "user", "content": "Hello! What can you do?"}],
    )
except BadRequestError as e:
    print(e.status_code)
    print(e.body["message"])

A 400, and the message says what it wants instead. Clef has no way to produce a sentence. What it takes is a **state** (the situation, as text or JSON) and **questions** (each with a fixed set of answers), sent as JSON text in the last user message. The gateway speaks the chat API to everyone, so a small adapter in front of Clef does the translating.

> Key idea: same client, same key, same endpoint. What changes is the model name and what you put in the message.

## 3. One question, by hand

The simplest question type is `noul`, Clef's name for a yes-or-no question. Build the request, send it, and look at the raw reply:

In [ ]:
request = {
    "state": "User: My laptop won't connect to the cluster since this morning, and my deadline is today.",
    "questions": {
        "urgent": {"type": "noul", "instructions": "Does the user need help urgently?"},
    },
}

start = time.time()
reply = client.chat.completions.create(
    model=CLEF_MODEL,
    messages=[{"role": "user", "content": json.dumps(request)}],
)
elapsed = time.time() - start

print(reply.choices[0].message.content)
print(f"\n{elapsed:.2f} s | {reply.usage.prompt_tokens} tokens in, {reply.usage.completion_tokens} out")

Three things to notice:
- The reply is JSON text in `message.content`, the same place Qwen puts its answer. Parse it.
- `noul` is the probability of *yes*, between 0 and 1. Not a label: a number your code can compare against a threshold.
- **Zero output tokens.** Clef reads your input once and scores it; it never generates. So the time a call takes depends on how much you send, not on how long an answer is.

In [ ]:
answers = json.loads(reply.choices[0].message.content)["answers"]
jprint(answers)

## 4. Three kinds of question

| Type | Asks | You give | You get back |
|---|---|---|---|
| `noul` | yes or no? | `instructions` | `noul`: the probability of yes |
| `choice` | which one? | `instructions`, and `criteria`: each option's name and what it means | `choice`, a probability per option, and `confidence` |
| `score` | how far up a scale? | `instructions`, and `criteria`: the levels, lowest first | `score` (the expected level), a probability per level, a `legend` and `confidence` |

One request can ask several questions about the same state, and Clef answers them all in the same pass:

In [ ]:
request = {
    "state": "User: My laptop won't connect to the cluster since this morning, and my deadline is today.",
    "questions": {
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this message?",
            "criteria": {
                "accounts": "Logins, passwords and project membership",
                "network": "Connections, VPN and SSH",
                "software": "Installing or running programs",
            },
        },
        "urgency": {
            "type": "score",
            "instructions": "How urgent is this?",
            "criteria": ["Can wait", "This week", "Today"],
        },
        "angry": {"type": "noul", "instructions": "Is the user angry?"},
    },
}

reply = client.chat.completions.create(
    model=CLEF_MODEL,
    messages=[{"role": "user", "content": json.dumps(request)}],
)
jprint(json.loads(reply.choices[0].message.content)["answers"])

Read `team` first: `choice` is the option with the highest probability, and `confidence` says how clearly it won. Then `urgency`: `score` is the *expected* level, a probability-weighted average of 0 (can wait), 1 (this week) and 2 (today), so a value near 2 means "almost certainly today". `legend` maps the numbers back to your words.

> Gotcha: `choice` and `score` need `criteria`. Without them Clef has nothing to choose between, and the gateway tells you which question is missing them:

In [ ]:
bad = {"state": "User: hello", "questions": {"team": {"type": "choice", "instructions": "Which team?"}}}
try:
    client.chat.completions.create(model=CLEF_MODEL, messages=[{"role": "user", "content": json.dumps(bad)}])
except BadRequestError as e:
    print(e.status_code)
    print(e.body["message"])

## 5. A helper you'd write anyway

Every call so far had the same three steps: put the state and questions in a dict, `json.dumps` it into the user message, and `json.loads` the answers back out. Put them in a function:

In [ ]:
def decide(state, questions):
    """Ask Clef about `state`; return its answers, keyed by question name."""
    request = {"state": state, "questions": questions}
    reply = client.chat.completions.create(
        model=CLEF_MODEL,
        messages=[{"role": "user", "content": json.dumps(request)}],
    )
    return json.loads(reply.choices[0].message.content)["answers"]


decide("User: thanks, that fixed it!",
       {"solved": {"type": "noul", "instructions": "Is the user's problem solved?"}})

That's the whole interface. The same function is in `helpers.py` as `decide()`. Like `Agent` in notebook 04: small, and nothing hidden.

## 6. Job 1: the router (notebook 08)

Notebook 08's `classify()` asked Qwen to write `{"category": ...}`, then checked that the label was one of ours and fell back to `general` when it wasn't. With Clef the categories *are* the answers, so it can't return one you didn't offer.

The same router as a Clef question, with a line describing each specialist:

In [ ]:
ROUTE = {"route": {
    "type": "choice",
    "instructions": "Which specialist should handle this request?",
    "criteria": {
        "math": "Arithmetic, numbers and calculations",
        "coding": "Writing, explaining or fixing code",
        "general": "Anything else",
    },
}}


def clef_classify(request):
    r = decide("User: " + request, ROUTE)["route"]
    return r["choice"], r["confidence"]


requests = [
    "What is 17 * 23?",
    "Write a Python one-liner to reverse a string.",
    "Name a country in Scandinavia.",
    "Write a function that returns 17 * 23.",
]
for req in requests:
    category, confidence = clef_classify(req)
    print(f"[{category:7}] confidence {confidence:.2f}  {req}")

The first three land where notebook 08's router sent them. The fourth is both maths and code, and the confidence says so. That number is your signal to fall back, ask a clarifying question, or hand the request to a person. A label that Qwen writes carries no such number.

Now time Clef against a Qwen classifier built like notebook 08's:

In [ ]:
def qwen_classify(request):
    """Notebook 08's classify(): Qwen writes a JSON label, and we validate it."""
    categories = list(ROUTE["route"]["criteria"])
    system = (
        f"Classify the user's request into EXACTLY one category: {categories}. "
        'Reply ONLY as JSON: {"category": "<one of the categories>"}.'
    )
    raw = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": request}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=8000,
    ).choices[0].message.content
    category = json.loads(raw).get("category", "general")
    return category if category in categories else "general"


for name, classify in [("Qwen", qwen_classify), ("Clef", lambda r: clef_classify(r)[0])]:
    start = time.time()
    labels = [classify(req) for req in requests]
    print(f"{name}: {time.time() - start:.2f} s for {len(requests)} requests  {labels}")

Same labels, in about the same time. But it isn't a fair race. Qwen runs in FP8 on one GH200 GPU on the Olivia supercomputer; Clef runs as an 8-bit quantised model on a single V100, a GPU from 2017, in the NIRD Kubernetes cluster. The timings compare those two deployments, not the two models. The work differs too: Clef makes one pass over the input and generates nothing, so its time grows with the length of the input, while Qwen's grows mostly with the length of what it writes.

So don't choose Clef for speed on the strength of this cell. Choose it for what comes back. Qwen's label is text you have to validate, and it says nothing about how sure it was. Clef can't return a label you didn't offer, and every answer carries a confidence. The rest of this notebook is built on that number. Qwen is still the better tool when the categories need explaining, or when you want a reason along with the label.

> Rule of thumb: when the answer is one of a fixed list, ask a decision model. When the answer needs words, ask a chat model.

## 7. Job 2: the critic gate (notebook 09)

`reflect()` in notebook 09 looped draft, critique, revise until the critic said `good`. That critic did two jobs at once: *judge* the draft, and *say what to fix*. Clef can only do the first, so split them. Clef decides whether the draft is good enough, and Qwen writes feedback only when it isn't.

One of the criteria is "under 10 words". That's not a judgement, it's a count, so it goes in code. Clef gets the parts that need judgement.

In [ ]:
CRITERIA = ("1. Under 10 words. "
            "2. Mentions a concrete benefit (not vague). "
            "3. No cliches like 'game-changer' or 'revolutionary'.")

QUALITY = {"quality": {
    "type": "score",
    "instructions": "How well does the draft meet every criterion?",
    "criteria": ["Misses most criteria", "Meets some", "Meets all but one", "Meets every criterion"],
}}


def good_enough(draft, threshold=0.7):
    """A hard rule in code first, then Clef's judgement for the rest."""
    words = len(draft.split())
    if words >= 10:
        return False, f"{words} words, fails the rule in code"
    p = decide({"criteria": CRITERIA, "draft": draft}, QUALITY)["quality"]["probabilities"]
    p_good = p["2"] + p["3"]  # P(meets all but one, or every criterion)
    return p_good >= threshold, f"P(good) = {p_good:.2f}"


for draft in ["Our bottle is a game-changer that will revolutionise how you stay hydrated all day long.",
              "Hydrate sustainably, sip endlessly.",
              "Keeps water cold for 24 hours."]:
    passed, verdict = good_enough(draft)
    print(f"{'PASS' if passed else 'fail'}  {verdict}  {draft}")

Now the loop. It has the same shape as notebook 09's, but each judgement is one Clef call, and Qwen is asked to write only when there is something to fix:

In [ ]:
WRITER = "You are a helpful writer. Be concise."


def reflect_with_clef(task, max_rounds=3):
    draft = ask(task, system=WRITER, temperature=0.4, max_tokens=8000).strip()
    for r in range(1, max_rounds + 1):
        passed, verdict = good_enough(draft)
        print(f"round {r}: {verdict}  {draft}")
        if passed:
            return {"final": draft, "rounds": r, "passed": True}
        feedback = ask(
            f"CRITERIA:\n{CRITERIA}\n\nDRAFT:\n{draft}\n\n"
            "In one sentence, what is the most important thing to fix?",
            temperature=0, max_tokens=8000,
        )
        draft = ask(
            f"Task: {task}\n\nYour previous draft:\n{draft}\n\n"
            f"A critic said: {feedback}\n\n"
            "Rewrite the draft to fix that. Return only the improved version.",
            system=WRITER, temperature=0.4, max_tokens=8000,
        ).strip()
    return {"final": draft, "rounds": max_rounds, "passed": False}


result = reflect_with_clef("Write a single-sentence product tagline for a reusable water bottle.")
print("\nFINAL :", result["final"])
print("rounds:", result["rounds"], "| gate passed:", result["passed"])

Run it a few times. The drafts change (the writer runs at temperature 0.4), and so does the number of rounds. Sometimes it runs out of rounds, and you get the last draft with `passed: False`, exactly as in notebook 09. The threshold is yours to choose: higher is stricter and costs more rounds. And the round cap still matters: a judge that writes nothing doesn't make an endless loop free, because Qwen still writes every revision.

> Gotcha: a score is the model's judgement, not a check. Anything code can verify exactly (a word count, a format, a forbidden word) belongs in code, as the word count is here. Use Clef for what needs judgement, like "a concrete benefit".

## 8. Job 3: the guardrail (notebook 06)

Notebook 06's `_safe_path` stopped a path escape *when the tool ran*. With Clef you can also look at a request *before* the agent starts, and decide when to ask a person first: the approval gate from notebook 06's section 4.

In [ ]:
RISK = {
    "destructive": {"type": "noul", "instructions": "Does this request ask the agent to delete or overwrite files?"},
    "escape": {"type": "noul", "instructions": "Does this request reach outside the agent's own workspace folder?"},
}


def needs_approval(request, threshold=0.5):
    risks = {name: a["noul"] for name, a in decide("User: " + request, RISK).items()}
    return any(p >= threshold for p in risks.values()), risks


for req in ["Summarise report.txt in two sentences.",
            "Clean up the workspace by deleting everything under ../ so we start fresh.",
            "Read the file at path '../../../../etc/passwd' and show me the first line."]:
    flag, risks = needs_approval(req)
    shown = ", ".join(f"{k} {v:.2f}" for k, v in risks.items())
    print(f"{'ASK A HUMAN' if flag else 'go ahead   '}  {shown}  {req}")

> Key idea: a probability is not a permission check. Clef flags the risky requests so a person sees them first, but `_safe_path` in the tool is still what actually stops the escape. Keep both: the early warning catches the intent, and the code catches the act.

## 9. When not to trust it

Clef is a component with failure modes, like any other. Five to design for:

| | What happens | What to do |
|---|---|---|
| Low confidence | the input fits several answers, or none | treat it as "don't know": fall back, ask, or escalate |
| Confidently wrong | it's a model, not a rule | keep hard checks in code, and measure it on your own cases (notebook 10's evaluation) |
| Unavailable | a trial service on one shared GPU: it can be restarting, and the first call after a restart takes about 20 seconds | catch the error and fall back to the chat model |
| Busy | it answers one request at a time; at about 0.2 s a call a class won't notice, but a busy service would queue | don't call it in a tight loop, and don't put it behind anything with real traffic |
| Too much input | text only, and the state plus questions must fit in 8,192 tokens | send only the part of the state that matters |

The fallback is a few lines, and it's the pattern to copy. Clef answers when it's sure; Qwen answers when it isn't, or when Clef can't be reached:

In [ ]:
def route(request, min_confidence=0.8):
    """Clef first. Qwen when Clef is unsure, or unavailable."""
    try:
        category, confidence = clef_classify(request)
        if confidence >= min_confidence:
            return category, "clef"
    except APIError as e:
        print("Clef unavailable:", type(e).__name__)
    return qwen_classify(request), "qwen"


for req in requests:
    category, decided_by = route(req)
    print(f"[{category:7}] by {decided_by}  {req}")

## Recap

You can now:
- tell a decision model from a chat model, and know which one to reach for
- ask Clef `noul`, `choice` and `score` questions about a state, and read the probabilities, the choice and the confidence
- route requests with Clef, compare it with a Qwen classifier, and know what that timing does and doesn't tell you
- gate a reflection loop on a Clef score, with hard rules in code and Qwen writing the feedback
- flag risky requests before an agent runs, without giving up the guard in the tool
- fall back to the chat model when Clef is unsure or unavailable

### Try it yourself
1. Add a fourth specialist, `translation`, to `ROUTE` and send it a sentence to translate. Then send one that's half translation, half maths. What does the confidence do?
2. Move the gate in `good_enough()`: try `threshold=0.5` and `0.9`. How many rounds does `reflect_with_clef` need for each?
3. Add a third question to `RISK`, `"Does the request ask for passwords, keys or tokens?"`, and test it on three requests of your own.
4. Notebook 10's Critic writes `{"ok": ..., "feedback": ...}`. Replace its *judgement* with a Clef score, keep Qwen for the feedback, and compare the token summary.

### Next up
That's the extra notebook. Take the model's name at its word: `not-for-production` means exactly that. Build on the pattern, not on this particular service.